# $d_\mathcal{X}$-privacy experiment reproduction

1. To set up the python environment, use `conda env create -n dxpriv --file dxpriv.yml`
2. In the data folder, put
    - glove.6B.300d.pkl extracted from glove.6B.zip downloaded from https://nlp.stanford.edu/projects/glove/
    - wiki.en.vec from English: bin+text downloaded from https://fasttext.cc/docs/en/pretrained-vectors.html

## Setup

In [1]:
from annoy import AnnoyIndex
import numpy as np
from numpy.random import normal
from os.path import join
import pickle
from collections import Counter
from datetime import datetime
from scipy.spatial import distance
# import cupy as cp
# from cupyx.scipy.spatial import distance
from concurrent.futures import ProcessPoolExecutor

data_folderpath = "./data/"

Import GloVe and Fasttext and keep GloVe embeddings of words in common

In [ ]:
with open(join(data_folderpath, "glove.6B.300d.pkl"), "rb") as f:
    glove = pickle.load(f)

with open(join(data_folderpath, "wiki.en.vec"), 'r', encoding='utf-8', newline='\n', errors='ignore') as f:
    n, d = map(int, f.readline().split())
    fasttext = {}
    for line in f:
        tokens = line.rstrip().split(' ')
        fasttext[tokens[0]] = map(float, tokens[1:])

common_words = set(fasttext.keys()) & set(glove.keys())
glove_commons = {key: glove[key] for key in common_words}

del glove; del fasttext # Saving RAM

In [10]:
vocab_embs = np.array(list(glove_commons.values()))
words_to_id = {word:index for index,word in enumerate(glove_commons.keys())}
id_to_words = list(glove_commons.keys())
vocab_size = vocab_embs.shape[0]
hidden_size = vocab_embs.shape[1]

del glove_commons # Saving RAM

In [11]:
# Code taken from https://github.com/awslabs/sagemaker-privacy-for-nlp/blob/master/source/sagemaker/src/package/data_privatization/data_privatization.py
def replace_word(sensitive_word, epsilon, ann_index, embedding_dims, sensitivity=1.0):
    """
    Given a word will inject noise according to the provided epsilon value and return a perturbed word.
    """
    # Generate a noise vector
    noise = generate_laplacian_noise_vector(embedding_dims, sensitivity, epsilon)
    # Get vector of sensitive word
    original_vec = vocab_embs[words_to_id[sensitive_word]]
    # Get perturbed vector
    noisy_vector = original_vec + noise
    # Get item closest to noisy vector
    closest_item = ann_index.get_nns_by_vector(noisy_vector, 1)[0]
    # Get word from item
    privatized_word = id_to_words[closest_item]
    return privatized_word


def generate_laplacian_noise_vector(dimension, sensitivity, epsilon):
    rand_vec = normal(size=dimension)
    normalized_vec = rand_vec / np.linalg.norm(rand_vec)
    magnitude = np.random.gamma(shape=dimension, scale=sensitivity / epsilon)
    return normalized_vec * magnitude

In [ ]:
# Index definition is the same as https://github.com/awslabs/sagemaker-privacy-for-nlp/blob/master/source/sagemaker/src/package/data_privatization/data_privatization.py
# Create approximate nearest neighbor index
num_trees = 50

ann_index = AnnoyIndex(hidden_size, 'euclidean')

for vector_num, vector in enumerate(vocab_embs):
    ann_index.add_item(vector_num, vector)

print("Building annoy index...")
assert ann_index.build(num_trees)
print("Annoy index built")
# Takes 2min

In [13]:
# This function is almost exactly the same as above, except that the exact nearest neighbor
# is found instead of leveraging the annoy index.
def replace_word_exact(sensitive_word, epsilon, embedding_dims, sensitivity=1.0):
    """
    Given a word will inject noise according to the provided epsilon value and return a perturbed word.
    """
    # Generate a noise vector
    noise = generate_laplacian_noise_vector(embedding_dims, sensitivity, epsilon)
    # Get vector of sensitive word
    original_vec = vocab_embs[words_to_id[sensitive_word]]
    # Get perturbed vector
    noisy_vector = original_vec + noise
    # Get item closest to noisy vector
    closest_item = distance.cdist(np.array([noisy_vector]), vocab_embs, 'euclidean').argmin(axis=-1).item()
    # Get word from item
    privatized_word = id_to_words[closest_item]
    return privatized_word

## Experiment 1

In [ ]:
word = "spacecraft"
epsilon = 19
repeat = 1000

# With Approximate Nearest Neighbor
privatized_word = [replace_word(word, epsilon, ann_index, hidden_size) for _ in range(repeat)]
print(Counter(privatized_word))

# With Exact Nearest Neighbor
privatized_word = [replace_word_exact(word, epsilon, hidden_size) for _ in range(repeat)]
print(Counter(privatized_word))

# Experiment 2

## With Approximate Nearest Neighbor

In [ ]:
epsilon = 43
repeat = 1000

def process_word(word):
    # Noise the word _repeat_ times and count when each result
    # is identical to the word.
    privatized_word = [replace_word(word, epsilon, ann_index, hidden_size) for _ in range(repeat)]
    return privatized_word.count(word)

# Parallelize the computation
with ProcessPoolExecutor(max_workers=26) as executor:
    results = list(executor.map(process_word, words_to_id.keys()))

# Compute the average
print(np.mean(results))
# 63 min

## With Exact Nearest Neighbor

In [ ]:
# ⚠ Extremely Slow
epsilon = 43
repeat = 1000

def process_word_exact(word):
    # Noise the word _repeat_ times and count when each result
    # is identical to the word.
    privatized_word = [replace_word_exact(word, epsilon, hidden_size) for _ in range(repeat)]
    return privatized_word.count(word)

# Parallelize the computation
with ProcessPoolExecutor(max_workers=26) as executor:
    results = list(executor.map(process_word_exact, words_to_id.keys()))

# Compute the average
print(np.mean(results))